# MammoDiffusion LDM v3 - architettura stile SD + v-prediction (+ min-SNR)

Terza iterazione del diffusore *from scratch*, pensata per essere un esperimento
**completo e affidabile** allo stesso livello di `04b2_LDM_SD_VAE_extra1361`: training,
generazione, filtro e valutazione usano tutti esplicitamente la stessa parameterization,
cosi' le metriche finali sono coerenti con come il modello e' stato addestrato.

Rispetto a `04b2_LDM_SD_VAE_extra1361` (latenti prodotti dal VAE preaddestrato di Stable
Diffusion 2.1 + U-Net Keras riaddestrata) cambiano tre cose:

1. **Architettura U-Net**: [`ldm_v3_unet_keras.py`](ldm_v3_unet_keras.py) sostituisce ogni `Conv2DTranspose` con `UpSampling2D(nearest) + Conv2D(3x3)`, adotta un `ResBlock` in stile SD (`GroupNorm -> SiLU -> Conv3x3`, embedding iniettato via FiLM) e usa `SiLU` come attivazione ovunque. Il downsampling resta `Conv2D stride=2` (gia' presente in v2).
2. **v-prediction** (Salimans & Ho 2022): la rete predice `v = sqrt(ab)*eps - sqrt(1-ab)*x0` invece del solo rumore. E' la parameterization usata da SD 2.1 e riduce gli artefatti a guidance scale elevata.
3. **Min-SNR-gamma weighting** (Hang et al. 2023): la loss simple viene pesata per campione con `w(t) = min(SNR(t), gamma) / SNR(t)` per eps o `min(SNR(t), gamma) / (SNR(t)+1)` per v, riducendo l'enfasi sui timestep a rumore basso (dove la loss e' facile) e accelerando la convergenza. Default `gamma=5`.

**Importante — coerenza eps/v-prediction:** `train_ldm_v2.py` addestra con `--parameterization v`,
quindi *tutte* le fasi successive (`evaluate_ldm_v2.py`, `generate_ldm_v2.py`) devono ricevere
esplicitamente `--parameterization v` (e `--unet-version v3`), altrimenti interpreterebbero
l'uscita v-prediction del modello come se fosse rumore puro, generando immagini sbagliate e
metriche non affidabili. Le celle di questo notebook passano questi flag esplicitamente invece
di riusare alla cieca le celle di `04b2` (che assumono `eps`): vedi la Sezione 9 in poi.

Il **VAE** puo' essere caricato in due modi:

* `USE_VAE_FT_FROM_03C = False` -> stesso VAE di 04b2 (SD-VAE congelato, `vae_source="sd_vae_original"`);
* `USE_VAE_FT_FROM_03C = True` -> VAE fine-tuned dal notebook `03c`, risolto automaticamente tra
  piu' path candidati (vedi Sezione 3). Se nessun candidato esiste, il notebook si ferma con un
  errore chiaro invece di ricadere silenziosamente sul VAE originale.

**Cartelle:** `experiments/diffusers/08_ldm_v3_sdvae_fromscratch/`, `results/diffusers/08_ldm_v3_sdvae_fromscratch/`,
immagini finali in `data/synthetic/fromscratch_v3/{positive, negative}/`.


## 1. Selezione GPU

In [ ]:
# GPU + XLA setup: deve essere eseguita prima di qualsiasi import TensorFlow.
#
# 04b3 e' pensato per la RTX 5060 Ti: la 3060 resta disponibile come fallback
# manuale, ma non viene piu' selezionata automaticamente.
# Con TensorFlow 2.15/CUDA 12.2 la 5060 Ti (Blackwell, compute capability 12.0)
# puo' mostrare warning di compilazione PTX. La parte importante e' fornire
# libdevice a XLA/MLIR e non far generare a TF kernel inutili prima del training.
import os
import subprocess
import sys
from pathlib import Path as _Path

PREFERRED_GPU_NAME = "5060"
FALLBACK_CUDA_VISIBLE_DEVICES = "0"

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"


def _query_nvidia_gpus():
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,uuid,name,pci.bus_id,compute_cap,memory.total",
        "--format=csv,noheader",
    ]
    try:
        proc = subprocess.run(cmd, check=False, capture_output=True, text=True)
    except Exception as exc:
        print(f"[WARN] nvidia-smi non disponibile: {exc}")
        return []
    if proc.returncode != 0:
        print("[WARN] nvidia-smi non ha restituito la lista GPU:")
        print((proc.stderr or proc.stdout).strip())
        return []

    rows = []
    for line in proc.stdout.splitlines():
        parts = [p.strip() for p in line.split(",", 5)]
        if len(parts) == 6:
            rows.append(
                {
                    "index": parts[0],
                    "uuid": parts[1],
                    "name": parts[2],
                    "bus": parts[3],
                    "cc": parts[4],
                    "memory": parts[5],
                }
            )
    return rows


def _select_gpu(preferred_name=PREFERRED_GPU_NAME):
    gpus = _query_nvidia_gpus()
    if gpus:
        print("GPU disponibili:")
        for gpu in gpus:
            marker = " <-- preferita" if preferred_name.lower() in gpu["name"].lower() else ""
            print(
                f"  index={gpu['index']} uuid={gpu['uuid']} name={gpu['name']} "
                f"cc={gpu['cc']} mem={gpu['memory']} bus={gpu['bus']}{marker}"
            )
        for gpu in gpus:
            if preferred_name.lower() in gpu["name"].lower():
                return gpu["uuid"]
        print(f"[WARN] Nessuna GPU contiene '{preferred_name}' nel nome; uso la prima GPU visibile.")
        return gpus[0]["uuid"]

    print(f"[WARN] Uso fallback CUDA_VISIBLE_DEVICES={FALLBACK_CUDA_VISIBLE_DEVICES}.")
    return FALLBACK_CUDA_VISIBLE_DEVICES


GPU_VISIBLE_DEVICES = _select_gpu()
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_VISIBLE_DEVICES

_libdevice_candidates = [
    _Path(sys.prefix) / "nvvm" / "libdevice" / "libdevice.10.bc",
    _Path("/home/fede/miniforge3/envs/tf-gpu/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/local/cuda/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/local/cuda-12.4/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/local/cuda-12.2/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/lib/nvidia-cuda-toolkit/nvvm/libdevice/libdevice.10.bc"),
]

_cuda_data_dir = None
for _candidate in _libdevice_candidates:
    if _candidate.exists():
        _cuda_data_dir = _candidate.parent.parent.parent
        break

if _cuda_data_dir is not None:
    os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={_cuda_data_dir}"
else:
    print("[WARN] libdevice.10.bc non trovato; se TF crasha su JIT, imposta XLA_FLAGS manualmente.")

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("TF_XLA_FLAGS:        ", os.environ.get("TF_XLA_FLAGS"))
print("XLA_FLAGS:           ", os.environ.get("XLA_FLAGS", "<non impostato>"))
print("[INFO] Se compare un warning su ptxas/CC 12.0, e' atteso con TF 2.15: il driver JIT fara' fallback.")


## 2. Setup dipendenze

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'pillow', 'gdown',
    'tensorflow', 'scikit-image', 'scipy', 'psutil', 'codecarbon',
    'torch', 'diffusers', 'transformers', 'safetensors', 'accelerate',
    'prdc', 'torch-fidelity',
])

## 3. Path progetto ed esperimento (dedicati a v3)

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import shutil
import zipfile
import time

import gdown

PROJECT_NAME = 'MammoDiffusion'
EXPERIMENT_NAME = 'diffusers/08_ldm_v3_sdvae_fromscratch'
RESULTS_STAGE_NAME = 'diffusers/08_ldm_v3_sdvae_fromscratch'
NOTEBOOK_NAME = '08_LDM_v3_SDVAE_FromScratch.ipynb'

SD21_MODEL_DRIVE_ID = '10XRn-bxpp7tP6ROWLYpeCNYJZIaHfMUt'
FORCE_MODEL_REDOWNLOAD = False
PROJECT_ROOT_OVERRIDE = None

# Flag espliciti passati a train_ldm_v2.py / generate_ldm_v2.py / evaluate_ldm_v2.py.
# Definiti qui una sola volta e riusati da tutte le celle sottostanti, cosi' non c'e'
# rischio che training e generazione/valutazione finiscano fuori sincrono.
UNET_VERSION = 'v3'
PARAMETERIZATION = 'v'
USE_MIN_SNR = True
MIN_SNR_GAMMA = 5.0

# NUOVO in 04b3: opzione per usare il VAE fine-tuned prodotto da 03c invece del VAE
# SD standard. Se True, la risoluzione del path e' robusta (piu' candidati) e fallisce
# con un errore esplicito se nessun candidato esiste (nessun fallback silenzioso).
USE_VAE_FT_FROM_03C = False
EXPERIMENT_03C_NAME = 'diffusers/03_sd21_vae_finetuned'


def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        return Path(override).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name or ((candidate / 'data').exists() and (candidate / 'notebooks').exists()):
            return candidate
    for candidate in [cwd / project_name, Path('/content') / project_name, Path.home() / project_name]:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError('Root MammoDiffusion non trovata.')


PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
UTILITY_DIR = NOTEBOOKS_DIR / 'utility'
DATA_DIR = PROJECT_ROOT / 'data'
DATA_PROCESSED_DIR = DATA_DIR / 'processed'
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / EXPERIMENT_NAME
EXPERIMENT_03C_DIR = PROJECT_ROOT / 'experiments' / EXPERIMENT_03C_NAME
SHARED_PRETRAINED_ROOT = NOTEBOOKS_DIR / 'pretrained_model'
PRETRAINED_MODEL_DIR = SHARED_PRETRAINED_ROOT / 'stable-diffusion-2-1-base'
PRETRAINED_MODEL_ZIP_PATH = SHARED_PRETRAINED_ROOT / 'archives' / 'stable-diffusion-2-1-base.zip'
MODELS_DIR = EXPERIMENT_DIR / 'models'
CHECKPOINTS_DIR = EXPERIMENT_DIR / 'checkpoints_ldm'
LATENTS_DIR = EXPERIMENT_DIR / 'latents'
LOGS_DIR = EXPERIMENT_DIR / 'logs'
RESULTS_DIR = PROJECT_ROOT / 'results' / RESULTS_STAGE_NAME
RESULTS_PLOTS_DIR = RESULTS_DIR / 'plots'
RESULTS_METRICS_DIR = RESULTS_DIR / 'metrics'
RESULTS_ECOTRACKER_DIR = RESULTS_DIR / 'ecotracker'
SYNTHETIC_V3_DIR = DATA_DIR / 'synthetic' / 'fromscratch_v3'
SYNTHETIC_V3_POS_DIR = SYNTHETIC_V3_DIR / 'positive'
SYNTHETIC_V3_NEG_DIR = SYNTHETIC_V3_DIR / 'negative'

for directory in [
    EXPERIMENT_DIR, PRETRAINED_MODEL_ZIP_PATH.parent, MODELS_DIR,
    CHECKPOINTS_DIR, LATENTS_DIR, LOGS_DIR,
    RESULTS_PLOTS_DIR, RESULTS_METRICS_DIR, RESULTS_ECOTRACKER_DIR,
    SYNTHETIC_V3_POS_DIR, SYNTHETIC_V3_NEG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def vae_ft_03c_candidate_dirs():
    """Path candidati (in ordine di preferenza) per il VAE fine-tuned di 03c. Non hardcoda
    un solo path: 03c puo' salvare il risultato in posizioni diverse a seconda che sia stato
    rilanciato con resume o meno."""
    return [
        EXPERIMENT_03C_DIR / 'vae_finetuning_resume_last' / 'vae_finetuned',
        EXPERIMENT_03C_DIR / 'vae_finetuning' / 'vae_finetuned',
        EXPERIMENT_03C_DIR / 'pretrained_model_vaeft' / 'stable-diffusion-2-1-base' / 'vae',
    ]


def resolve_vae_ft_03c_dir():
    """Restituisce il primo candidato che contiene un config.json (formato diffusers),
    oppure None se nessuno e' valido. Nessun fallback silenzioso: la decisione su cosa fare
    in assenza di un candidato valido spetta al chiamante."""
    for candidate in vae_ft_03c_candidate_dirs():
        if (candidate / 'config.json').is_file():
            return candidate
    return None


print('PROJECT_ROOT   :', PROJECT_ROOT)
print('EXPERIMENT_DIR :', EXPERIMENT_DIR)
print('RESULTS_DIR    :', RESULTS_DIR)
print('SYNTHETIC v3   :', SYNTHETIC_V3_DIR)
print('UNET_VERSION   :', UNET_VERSION)
print('PARAMETERIZATION:', PARAMETERIZATION)
print('USE_MIN_SNR    :', USE_MIN_SNR, '| MIN_SNR_GAMMA:', MIN_SNR_GAMMA)
print('USE_VAE_FT_FROM_03C:', USE_VAE_FT_FROM_03C)


## 4. Verifica dataset preprocessato (identica a 04b2)

In [ ]:
import pandas as pd

required = [
    DATA_PROCESSED_DIR / 'metadata' / 'train.csv',
    DATA_PROCESSED_DIR / 'metadata' / 'val.csv',
    DATA_PROCESSED_DIR / 'metadata' / 'test.csv',
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Dataset preprocessato mancante: {path}')

for split in ['train', 'val', 'test']:
    df = pd.read_csv(DATA_PROCESSED_DIR / 'metadata' / f'{split}.csv')
    print(split, len(df), df['label'].value_counts().to_dict())

## 5. Modello SD2.1 base (per il VAE)

Scarica il modello base se assente. Se `USE_VAE_FT_FROM_03C=True`, sostituisce la
sottocartella `vae/` con il VAE fine-tuned di 03c prima di procedere.

In [ ]:
MODEL_WEIGHT_ALIASES = {
    'text_encoder': ('model.fp16.safetensors', 'model.safetensors'),
    'unet':         ('diffusion_pytorch_model.fp16.safetensors', 'diffusion_pytorch_model.safetensors'),
    'vae':          ('diffusion_pytorch_model.fp16.safetensors', 'diffusion_pytorch_model.safetensors'),
}


def has_diffusers_structure(d):
    return all((Path(d) / p).exists() for p in ['model_index.json', 'scheduler', 'tokenizer', 'text_encoder', 'vae', 'unet'])


def create_model_weight_copies(model_dir: Path):
    for subfolder, (source_name, target_name) in MODEL_WEIGHT_ALIASES.items():
        source_path = model_dir / subfolder / source_name
        target_path = model_dir / subfolder / target_name
        if target_path.exists() or not source_path.is_file():
            continue
        shutil.copy2(source_path, target_path)


def prepare_sd_model():
    if has_diffusers_structure(PRETRAINED_MODEL_DIR) and not FORCE_MODEL_REDOWNLOAD:
        print('Modello SD2.1 gia\' presente.')
        create_model_weight_copies(PRETRAINED_MODEL_DIR)
        return
    if not PRETRAINED_MODEL_ZIP_PATH.exists():
        gdown.download(id=SD21_MODEL_DRIVE_ID, output=str(PRETRAINED_MODEL_ZIP_PATH), quiet=False)
    with TemporaryDirectory(prefix='sd21_extract_') as tmp:
        with zipfile.ZipFile(PRETRAINED_MODEL_ZIP_PATH) as archive:
            archive.extractall(tmp)
        for path in Path(tmp).rglob('model_index.json'):
            src_dir = path.parent
            if PRETRAINED_MODEL_DIR.exists():
                shutil.rmtree(PRETRAINED_MODEL_DIR)
            shutil.copytree(src_dir, PRETRAINED_MODEL_DIR)
            create_model_weight_copies(PRETRAINED_MODEL_DIR)
            break

prepare_sd_model()

# Opzionalmente sostituisce il VAE con la variante fine-tuned di 03c. A differenza di
# un singolo path hardcoded, resolve_vae_ft_03c_dir() prova piu' candidati (vedi Sezione 3)
# e solleva un errore chiaro se USE_VAE_FT_FROM_03C=True ma nessuno e' disponibile,
# invece di ricadere silenziosamente sul VAE originale.
if USE_VAE_FT_FROM_03C:
    vae_ft_dir = resolve_vae_ft_03c_dir()
    if vae_ft_dir is None:
        candidates_str = '\n  - '.join(str(p) for p in vae_ft_03c_candidate_dirs())
        raise FileNotFoundError(
            'USE_VAE_FT_FROM_03C=True ma non ho trovato il VAE fine-tuned di 03c in nessuno '
            f'dei path candidati:\n  - {candidates_str}\n'
            'Esegui prima 03_SD21_VAE_FineTuned.ipynb oppure imposta '
            'USE_VAE_FT_FROM_03C=False.'
        )
    dst = PRETRAINED_MODEL_DIR / 'vae'
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(vae_ft_dir, dst)
    create_model_weight_copies(PRETRAINED_MODEL_DIR)
    print('VAE fine-tuned di 03c copiato da:', vae_ft_dir, '->', dst)
    VAE_SOURCE = 'sd_vae_finetuned_03c'
else:
    VAE_SOURCE = 'sd_vae_original'

SD_VAE_MODEL = PRETRAINED_MODEL_DIR
print('SD_VAE_MODEL:', SD_VAE_MODEL)
print('VAE_SOURCE  :', VAE_SOURCE)


## 6. Encoding latenti con SD-VAE (o VAE fine-tuned di 03c)

Riusa lo stesso script di 04b2 (`prepare_sdvae_latents_v2.py`). Se `USE_VAE_FT_FROM_03C=True`,
lo script legge il VAE che abbiamo appena copiato nella sotto-cartella `vae/` del modello base.

In [ ]:
def run_and_stream(cmd, log_path):
    print(' '.join(str(c) for c in cmd))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        for line in iter(process.stdout.readline, ''):
            handle.write(line)
            print(line, end='', flush=True)
    process.wait()
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)


SDVAE_BATCH_SIZE = 4
FORCE_LATENTS_RECOMPUTE = False

prepare_cmd = [
    sys.executable,
    str(UTILITY_DIR / 'prepare_sdvae_latents_v2.py'),
    '--project-root', str(PROJECT_ROOT),
    '--experiment-dir', str(EXPERIMENT_DIR),
    '--batch-size', str(SDVAE_BATCH_SIZE),
    '--results-stage-name', RESULTS_STAGE_NAME,
    '--sd-vae-model', str(SD_VAE_MODEL),
]
if GPU_VISIBLE_DEVICES is not None:
    prepare_cmd.extend(['--gpu-visible-devices', GPU_VISIBLE_DEVICES])
if FORCE_LATENTS_RECOMPUTE:
    prepare_cmd.append('--force-recompute')

run_and_stream(prepare_cmd, LOGS_DIR / 'prepare_sdvae_latents.log')


## 7. Training U-Net v3 (v-prediction + min-SNR)

Lancia `train_ldm_v2.py` con i flag v3 definiti nella Sezione 3 (`UNET_VERSION`,
`PARAMETERIZATION`, `USE_MIN_SNR`, `MIN_SNR_GAMMA`), piu' `--vae-source` e
`--uses-vae-ft-from-03c` per tracciare nel manifest quale VAE e' stato usato:

* `--unet-version v3` -> il builder importa `ldm_v3_unet_keras.build_ldm_unet_v3`;
* `--parameterization v` -> loss simple = `||v_target - v_pred||^2`;
* `--use-min-snr --min-snr-gamma 5.0` -> weighting Min-SNR-gamma (formula corretta per v-prediction);
* `--vae-source` / `--uses-vae-ft-from-03c` -> salvati in `training_manifest.json`, letto
  automaticamente da generazione/valutazione/04c.

Al termine del training, `train_ldm_v2.py` scrive sempre `training_manifest.json` in
`EXPERIMENT_DIR`, cosi' generazione e valutazione (e il notebook 04c) possono rilevare
la parameterization invece di doverla assumere.

Manteniamo gli stessi ordini di grandezza di 04b2 (150k step) per confrontabilita' diretta.


In [ ]:
TOTAL_STEPS = 150_000
CHECKPOINT_EVERY = 5_000
LOG_EVERY = 20
RESUME_FROM_LATEST = True

train_cmd = [
    sys.executable,
    str(UTILITY_DIR / 'train_ldm_v2.py'),
    '--project-root', str(PROJECT_ROOT),
    '--experiment-dir', str(EXPERIMENT_DIR),
    '--total-steps', str(TOTAL_STEPS),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--log-every', str(LOG_EVERY),
    '--results-stage-name', RESULTS_STAGE_NAME,
    '--skip-latent-encoding',
    '--unet-version', UNET_VERSION,
    '--parameterization', PARAMETERIZATION,
    '--vae-source', VAE_SOURCE,
    '--notebook-name', NOTEBOOK_NAME,
]
if USE_MIN_SNR:
    train_cmd.append('--use-min-snr')
    train_cmd.extend(['--min-snr-gamma', str(MIN_SNR_GAMMA)])
if USE_VAE_FT_FROM_03C:
    train_cmd.append('--uses-vae-ft-from-03c')
if RESUME_FROM_LATEST:
    train_cmd.append('--resume-from-latest')

run_and_stream(train_cmd, LOGS_DIR / 'ldm_train_v3.log')


## 8. Sweep dei checkpoint, generazione, filtro e valutazione finale

Le celle seguenti richiamano `evaluate_ldm_v2.py` e `generate_ldm_v2.py` con la stessa
logica di `04b2`, ma **non** riusano le celle di `04b2` per exec dinamico: sono celle
autonome che passano esplicitamente `--unet-version v3 --parameterization v` (e
`--vae-source` / `--uses-vae-ft-from-03c`), cosi' non c'e' rischio che una futura modifica
a `04b2` cambi silenziosamente il comportamento di `04b3`, e non c'e' rischio che venga
dimenticata la conversione v-prediction -> epsilon durante generazione/valutazione.

Ogni comando include anche `--notebook-name` per tracciabilita' nei manifest/JSON prodotti.


### 8.1 Sweep checkpoint (FID/IS su validation)

In [ ]:
EVAL_MIN_STEP = 1_000
N_GEN_PER_CLASS = 100
EVAL_SAMPLE_STEPS = 100
EVAL_GUIDANCE_SCALE = 1.5
EVAL_INCEPTION_BATCH = 8
EVAL_DECODE_ON_CPU = False
EVAL_ECO_TRACK = True

eval_cmd = [
    sys.executable,
    str(UTILITY_DIR / "evaluate_ldm_v2.py"),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--mode", "both",
    "--min-step", str(EVAL_MIN_STEP),
    "--n-gen-per-class", str(N_GEN_PER_CLASS),
    "--sample-steps", str(EVAL_SAMPLE_STEPS),
    "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
    "--mini-batch", "1",
    "--inception-batch", str(EVAL_INCEPTION_BATCH),
    "--results-stage-name", RESULTS_STAGE_NAME,
    "--vae-backend", "sd",
    "--sd-vae-model", str(SD_VAE_MODEL),
    "--unet-version", UNET_VERSION,
    "--parameterization", PARAMETERIZATION,
    "--vae-source", VAE_SOURCE,
    "--notebook-name", NOTEBOOK_NAME,
]
if USE_VAE_FT_FROM_03C:
    eval_cmd.append("--uses-vae-ft-from-03c")
if EVAL_DECODE_ON_CPU:
    eval_cmd.append("--decode-on-cpu")
if EVAL_ECO_TRACK:
    eval_cmd.append("--eco-track")

run_and_stream(eval_cmd, LOGS_DIR / "ldm_evaluate_sdvae_v3.log")


### 8.1b Ottimizzazione dei parametri di sampling sul best checkpoint

Sostituisce il vecchio notebook standalone `04d_LDM_Optimized.ipynb`: qui
l'ottimizzazione viene fatta direttamente sopra il best checkpoint di questa
LDM v3 from-scratch (che era l'ultimo test di modello from-scratch previsto).

Idea: fissato il checkpoint scelto in 10.1, si stima *quanti* step di DDIM sono
davvero necessari per raggiungere la qualita' che la cella 10.1 ha misurato a
100 step. Un numero di step piu' basso a parita' di FID accelera la generazione
finale (10.2/10.3) e la valutazione, riducendo tempo e CO2.

Cosa fa la cella successiva:

1. Legge `evaluation/checkpoint_metrics.json` prodotto da 10.1, individua il
   `best_step` scelto (quello con `avg_FID` minimo).
2. Per ogni budget in `SAMPLING_STEP_BUDGETS` (default 25, 50, 75, 100), lancia
   `evaluate_ldm_v2.py` sul solo `step_{best_step}` con `--sample-steps N` e
   `--n-gen-per-class 50` (poche immagini bastano per una stima di FID
   affidabile in confronto relativo). L'output `checkpoint_metrics.json` viene
   spostato in `evaluation/sampling_sweep/steps_{N}_metrics.json` per non
   sovrascrivere quello di 10.1.
3. Aggrega i risultati in un DataFrame e salva
   `sampling_sweep_summary.csv`, senza generare plot dedicati.
4. Suggerisce `GEN_SAMPLE_STEPS_RECOMMENDED`: il minimo budget che raggiunge
   FID <= (FID a 100 step) * `SAMPLING_QUALITY_TOLERANCE` (default 1.05).
   Le celle 10.2/10.3 continuano a usare `GEN_SAMPLE_STEPS = 100` come default
   sicuro; e' la sezione 10.1b a dire se scendere e a quale valore.

La sezione e' **skippata automaticamente** se `checkpoint_metrics.json` non
esiste ancora (10.1 non e' stata eseguita) o se `RUN_SAMPLING_SWEEP = False`.


In [ ]:
# Ottimizzazione sampler integrata in 04b3 (era il vecchio 04d, eliminato).
# Riusa evaluate_ldm_v2.py, quindi zero nuova logica di caricamento modello.

RUN_SAMPLING_SWEEP = True
SAMPLING_STEP_BUDGETS = [25, 50, 75, 100]
SAMPLING_N_GEN_PER_CLASS = 50   # 50 immagini/classe bastano per un confronto relativo affidabile
SAMPLING_QUALITY_TOLERANCE = 1.05  # accetta un budget se FID <= 1.05 * FID(100 step)

evaluation_dir = EXPERIMENT_DIR / "evaluation"
checkpoint_metrics_path = evaluation_dir / "checkpoint_metrics.json"
sampling_sweep_dir = evaluation_dir / "sampling_sweep"
sampling_sweep_dir.mkdir(parents=True, exist_ok=True)
sampling_sweep_csv = sampling_sweep_dir / "sampling_sweep_summary.csv"

GEN_SAMPLE_STEPS_RECOMMENDED = None

if not RUN_SAMPLING_SWEEP:
    print("Sampling sweep disattivato (RUN_SAMPLING_SWEEP=False): skip.")
elif not checkpoint_metrics_path.is_file():
    print(f"checkpoint_metrics.json non trovato in {checkpoint_metrics_path}. "
          "Esegui prima la cella 10.1, poi torna qui.")
else:
    import pandas as pd

    with checkpoint_metrics_path.open(encoding="utf-8") as handle:
        eval_payload = json.load(handle)

    # eval_payload["candidates"] contiene le metriche per ogni checkpoint valutato.
    # Selezione best = min avg_FID (stessa logica di evaluate_ldm_v2 --mode both).
    candidates = eval_payload.get("candidates", [])
    step_candidates = [c for c in candidates if c.get("checkpoint_id", "").startswith("step_")]
    if not step_candidates:
        raise RuntimeError("Nessun step-checkpoint valutato in checkpoint_metrics.json.")
    best_row = min(step_candidates, key=lambda c: c["metrics"]["avg_FID"])
    best_checkpoint_id = best_row["checkpoint_id"]
    best_step = int(best_checkpoint_id.replace("step_", ""))
    fid_at_100 = float(best_row["metrics"]["avg_FID"])
    print(f"Best checkpoint da 10.1: {best_checkpoint_id} (avg_FID @ {EVAL_SAMPLE_STEPS} step = {fid_at_100:.3f})")

    records = []
    for steps in SAMPLING_STEP_BUDGETS:
        target_metrics_json = sampling_sweep_dir / f"steps_{steps}_metrics.json"
        if target_metrics_json.is_file():
            print(f"[{steps} step] gia' calcolato -> riuso {target_metrics_json.name}")
        else:
            print(f"[{steps} step] eseguo evaluate_ldm_v2.py --checkpoint-id {best_checkpoint_id} --sample-steps {steps}")
            sweep_eval_cmd = [
                sys.executable,
                str(UTILITY_DIR / "evaluate_ldm_v2.py"),
                "--project-root", str(PROJECT_ROOT),
                "--experiment-dir", str(EXPERIMENT_DIR),
                "--mode", "evaluate",
                "--checkpoint-id", best_checkpoint_id,
                "--n-gen-per-class", str(SAMPLING_N_GEN_PER_CLASS),
                "--sample-steps", str(steps),
                "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
                "--mini-batch", "1",
                "--inception-batch", str(EVAL_INCEPTION_BATCH),
                "--results-stage-name", f"{RESULTS_STAGE_NAME}_sampling_sweep_{steps}",
                "--vae-backend", "sd",
                "--sd-vae-model", str(SD_VAE_MODEL),
                "--unet-version", UNET_VERSION,
                "--parameterization", PARAMETERIZATION,
                "--vae-source", VAE_SOURCE,
                "--notebook-name", NOTEBOOK_NAME,
                "--force-recompute",
            ]
            if USE_VAE_FT_FROM_03C:
                sweep_eval_cmd.append("--uses-vae-ft-from-03c")
            run_and_stream(sweep_eval_cmd, LOGS_DIR / f"sampling_sweep_{steps}steps.log")
            if not checkpoint_metrics_path.is_file():
                raise RuntimeError("evaluate_ldm_v2.py non ha prodotto checkpoint_metrics.json.")
            shutil.move(checkpoint_metrics_path, target_metrics_json)

        with target_metrics_json.open(encoding="utf-8") as handle:
            payload = json.load(handle)
        row = next(c for c in payload["candidates"] if c["checkpoint_id"] == best_checkpoint_id)
        m = row["metrics"]
        wall_time = payload.get("wall_time_seconds") or row.get("wall_time_seconds", 0.0)
        records.append({
            "sample_steps": steps,
            "avg_FID": float(m["avg_FID"]),
            "avg_IS_mean": float(m.get("avg_IS_mean", float("nan"))),
            "avg_precision": float(m.get("avg_precision", float("nan"))),
            "avg_recall": float(m.get("avg_recall", float("nan"))),
            "seconds_per_image": float(wall_time) / max(1, 2 * SAMPLING_N_GEN_PER_CLASS) if wall_time else float("nan"),
        })

    df_sweep = pd.DataFrame(records).sort_values("sample_steps").reset_index(drop=True)
    df_sweep.to_csv(sampling_sweep_csv, index=False)
    print("\nRiepilogo sampling sweep:")
    print(df_sweep.to_string(index=False))

    print("Plot sampling sweep disattivati: metriche tabellari salvate in", sampling_sweep_csv)

    # Raccomandazione: minimo budget con FID entro tolleranza rispetto al reference @ 100 step.
    tol_target = fid_at_100 * SAMPLING_QUALITY_TOLERANCE
    acceptable = df_sweep[df_sweep["avg_FID"] <= tol_target]
    if acceptable.empty:
        GEN_SAMPLE_STEPS_RECOMMENDED = int(df_sweep["sample_steps"].iloc[-1])
        print(f"\nNessun budget entro tolleranza {SAMPLING_QUALITY_TOLERANCE:.0%} vs FID @ {EVAL_SAMPLE_STEPS} step ({fid_at_100:.3f}): "
              f"resta consigliato il massimo ({GEN_SAMPLE_STEPS_RECOMMENDED} step).")
    else:
        GEN_SAMPLE_STEPS_RECOMMENDED = int(acceptable["sample_steps"].min())
        best_row_sweep = acceptable.iloc[0]
        print(f"\nRACCOMANDAZIONE: usa {GEN_SAMPLE_STEPS_RECOMMENDED} step "
              f"(FID {float(best_row_sweep['avg_FID']):.3f} vs riferimento {fid_at_100:.3f}, "
              f"entro {SAMPLING_QUALITY_TOLERANCE:.0%}). "
              f"Modifica GEN_SAMPLE_STEPS nelle celle 10.2/10.3 se accetti la raccomandazione.")

    with (sampling_sweep_dir / "recommendation.json").open("w", encoding="utf-8") as handle:
        json.dump({
            "best_checkpoint_id": best_checkpoint_id,
            "best_step": int(best_step),
            "reference_sample_steps": int(EVAL_SAMPLE_STEPS),
            "reference_fid": float(fid_at_100),
            "quality_tolerance": float(SAMPLING_QUALITY_TOLERANCE),
            "step_budgets_tested": list(SAMPLING_STEP_BUDGETS),
            "recommended_gen_sample_steps": (int(GEN_SAMPLE_STEPS_RECOMMENDED)
                                             if GEN_SAMPLE_STEPS_RECOMMENDED is not None else None),
            "sweep_summary_csv": str(sampling_sweep_csv),
        }, handle, indent=2, ensure_ascii=False)


### 8.2 Generazione, filtro e test - classe positiva

In [ ]:
GEN_N_RAW = 4083
GEN_N_SELECTED = 1361
GEN_SAMPLE_STEPS = 100
GEN_GUIDANCE_SCALE = 1.5
GEN_MODEL_PATH = CHECKPOINTS_DIR / "ldm_unet_best_eval.keras"
GEN_DECODE_ON_CPU = False
GEN_ECO_TRACK = True

pos_cmd = [
    sys.executable,
    str(UTILITY_DIR / "generate_ldm_v2.py"),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--model-path", str(GEN_MODEL_PATH),
    "--mode", "all",
    "--n-raw", str(GEN_N_RAW),
    "--n-selected", str(GEN_N_SELECTED),
    "--target-label", "1",
    "--batch-size", "1",
    "--sample-steps", str(GEN_SAMPLE_STEPS),
    "--guidance-scale", str(GEN_GUIDANCE_SCALE),
    "--results-stage-name", RESULTS_STAGE_NAME,
    "--vae-backend", "sd",
    "--sd-vae-model", str(SD_VAE_MODEL),
    "--unet-version", UNET_VERSION,
    "--parameterization", PARAMETERIZATION,
    "--vae-source", VAE_SOURCE,
    "--notebook-name", NOTEBOOK_NAME,
]
if USE_VAE_FT_FROM_03C:
    pos_cmd.append("--uses-vae-ft-from-03c")
if GEN_DECODE_ON_CPU:
    pos_cmd.append("--decode-on-cpu")
if GEN_ECO_TRACK:
    pos_cmd.append("--eco-track")

# "--mode all" scrive nei path canonici dell'esperimento (get_experiment_paths), che per
# EXPERIMENT_NAME=diffusers/08_ldm_v3_sdvae_fromscratch risolvono a SYNTHETIC_V3_POS_DIR
# (vedi ldm_project_paths.FILTERED_DIR_NAME_BY_EXPERIMENT) -- niente collisione con
# 04b/04b2.
run_and_stream(pos_cmd, LOGS_DIR / "ldm_generate_positive_sdvae_v3.log")


### 8.3 Generazione, filtro, validazione e test - classe negativa


In [ ]:
NEG_RAW_DIR = EXPERIMENT_DIR / "synthetic_raw_negative"
NEG_FILTERED_DIR = SYNTHETIC_V3_NEG_DIR

neg_base_cmd = [
    sys.executable,
    str(UTILITY_DIR / "generate_ldm_v2.py"),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--model-path", str(GEN_MODEL_PATH),
    "--n-raw", str(GEN_N_RAW),
    "--n-selected", str(GEN_N_SELECTED),
    "--target-label", "0",
    "--raw-dir", str(NEG_RAW_DIR),
    "--filtered-dir", str(NEG_FILTERED_DIR),
    "--batch-size", "1",
    "--sample-steps", str(GEN_SAMPLE_STEPS),
    "--guidance-scale", str(GEN_GUIDANCE_SCALE),
    "--results-stage-name", RESULTS_STAGE_NAME,
    "--vae-backend", "sd",
    "--sd-vae-model", str(SD_VAE_MODEL),
    "--unet-version", UNET_VERSION,
    "--parameterization", PARAMETERIZATION,
    "--vae-source", VAE_SOURCE,
    "--notebook-name", NOTEBOOK_NAME,
]
if USE_VAE_FT_FROM_03C:
    neg_base_cmd.append("--uses-vae-ft-from-03c")
if GEN_DECODE_ON_CPU:
    neg_base_cmd.append("--decode-on-cpu")
if GEN_ECO_TRACK:
    neg_base_cmd.append("--eco-track")

neg_cmd = [*neg_base_cmd, "--mode", "all"]
run_and_stream(neg_cmd, LOGS_DIR / "ldm_negative_sdvae_v3_all.log")


### 8.4 Riepilogo artefatti


In [ ]:
import json as _json
from pathlib import Path
import pandas as pd

summary = {
    "best_eval": CHECKPOINTS_DIR / "ldm_unet_best_eval.keras",
    "training_manifest": EXPERIMENT_DIR / "training_manifest.json",
    "latent_stats": LATENTS_DIR / "latent_stats.npz",
    "positive_final_json": RESULTS_METRICS_DIR / "positive" / "final_filtered_vs_test.json",
    "negative_final_json": RESULTS_METRICS_DIR / "negative" / "final_filtered_vs_test.json",
    "positive_filtered_dir": SYNTHETIC_V3_POS_DIR,
    "negative_filtered_dir": SYNTHETIC_V3_NEG_DIR,
}
for name, path in summary.items():
    print(f"{name:22s}", path, "OK" if Path(path).exists() else "MISSING")

if summary["training_manifest"].exists():
    with open(summary["training_manifest"], encoding="utf-8") as handle:
        training_manifest = _json.load(handle)
    print("\ntraining_manifest.json:")
    print(_json.dumps(training_manifest, indent=2, ensure_ascii=False))

rows = []
for label_name, directory in [("positive", SYNTHETIC_V3_POS_DIR), ("negative", SYNTHETIC_V3_NEG_DIR)]:
    rows.append({
        "class": label_name,
        "directory": str(directory),
        "n_png": len(sorted(Path(directory).glob("*.png"))) if Path(directory).is_dir() else 0,
    })
pd.DataFrame(rows)
